In [1]:
import numpy as np
from astropy.io import fits
from astropy.wcs import WCS
from astropy.coordinates import SkyCoord
import astropy.units as u
import mysql.connector
import pandas as pd
import os

In [2]:
mydb = mysql.connector.connect(
  host="127.0.0.1",
  user="root",
  #passwd=" OpenSQL1000",        #<---- 
  database = "HiGAL_250_280"     #<---- 
)

In [5]:
def extract(cube_file, output_dir):
    # Open the spectral cube FITS file
    with fits.open(cube_file,ignore_missing_simple=True) as hdul:
        cube_data = hdul[0].data
        header = hdul[0].header
        wcs = WCS(header)

    # Create SQL engine and read coordinates -- crates data frame with all clumps within the cube
	# This is set for the 1 degree cubes. The range will need to be changed if bigger
    df = pd.read_sql_query(f'SELECT gal_long, gal_lat, higal_name FROM higal_mopra_co WHERE gal_long>{int(cube_file[1:4])} and gal_long<{int(cube_file[1:4])+10}', con=mydb)
    #df = pd.read_sql_query(f'SELECT gal_long, gal_lat, higal_name FROM higal_mopra_co WHERE higal_name="HIGALBM259.9911-0.2148"', con=mydb)

    for index, row in df.iterrows():
        # Convert galactic coordinates to pixel coordinates
        sky_coord = SkyCoord(l=row['gal_long']*u.deg, b=row['gal_lat']*u.deg, frame='galactic')
        x, y = sky_coord.to_pixel(wcs, origin=0)

        # Define the 3x3 pixel area around the source
        x_min_spec, x_max_spec = int(x) - 1, int(x) + 1
        y_min_spec, y_max_spec = int(y) - 1, int(y) + 1

        # Ensure we don't go out of bounds for the spectra
        x_min_spec = max(x_min_spec, 0)
        y_min_spec = max(y_min_spec, 0)
        x_max_spec = min(x_max_spec, cube_data.shape[2] - 1)
        y_max_spec = min(y_max_spec, cube_data.shape[1] - 1)

        # Extract the smaller cube for the spectra
        subcube_spec = cube_data[:, y_min_spec:y_max_spec+1, x_min_spec:x_max_spec+1]

        # Compute the spectra (average over the 3x3 area)
        spectra = np.mean(subcube_spec, axis=(1, 2))

        # Create a new FITS file for the spectra
        hdu_spec = fits.PrimaryHDU(spectra, header)
        output_path_spec = f"{output_dir}/12CO_{row['higal_name']}_spec.fits"
       # hdu_spec.writeto(output_path_spec, overwrite=True)
        #print(f"Saved spectra to {output_path_spec}")


In [52]:
output_dir = '/Users/kyriakos.trakakis/Desktop/Temp'		# output directory
cube_folder = os.listdir('/Users/kyriakos.trakakis/Downloads/MOPRA_HiGAL' )	  	# directory of cubes
# spectra_folder.remove('.DS_Store')
cube_folder.sort()

In [53]:
cube_folder

['.DS_Store',
 '.ipynb_checkpoints',
 'Extract_spectra_from_cube.ipynb',
 'G250-260-12CO_giantcube_fullres.fits',
 'G250-260-13CO_giantcube_fullres.fits',
 'G250-260-C18O_giantcube_fullres.fits',
 'G260-270-12CO_giantcube_fullres.fits',
 'G260-270-13CO_giantcube_fullres.fits',
 'G260-270-C18O_giantcube_fullres.fits',
 'G270-280-12CO_giantcube_fullres.fits',
 'G270-280-13CO_giantcube_fullres.fits',
 'G270-280-C18O_giantcube_fullres.fits',
 'HiGAL_250_280.csv',
 'HiGAL_250_280.sql',
 'Low_Res',
 'Ploting_Spectra_MOPRA.py',
 'Plotting_spectra_Mopra.ipynb',
 'README.txt',
 'extract_spec_from_cube.py',
 'spectra_png']

In [54]:
cube_file_1='G250-260-12CO_giantcube_fullres.fits'
cube_file_2='G260-270-12CO_giantcube_fullres.fits'
cube_file_3='G270-280-12CO_giantcube_fullres.fits'

In [55]:
extract(cube_file_1, output_dir)
#extract(cube_file_2, output_dir)
#extract(cube_file_3, output_dir)

/var/folders/my/spqrhq7n74bcr2n7xfhc9hhr0000gp/T/ipykernel_87379/201573063.py:11: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  df = pd.read_sql_query(f'SELECT gal_long, gal_lat, higal_name FROM higal_mopra_co', con=mydb)
/Users/kyriakos.trakakis/anaconda3/lib/python3.11/site-packages/numpy/core/fromnumeric.py:3504: RuntimeWarning: Mean of empty slice.
  return _methods._mean(a, axis=axis, dtype=dtype,
/Users/kyriakos.trakakis/anaconda3/lib/python3.11/site-packages/numpy/core/_methods.py:121: RuntimeWarning: invalid value encountered in divide
  ret = um.true_divide(
/Users/kyriakos.trakakis/anaconda3/lib/python3.11/site-packages/numpy/core/fromnumeric.py:3504: RuntimeWarning: Mean of empty slice.
  return _methods._mean(a, axis=axis, dtype=dtype,
/Users/kyriakos.trakakis/anaconda3/lib/python3.11/site-packages/numpy/core/_methods.p

In [34]:
int(cube_file_1[1:4])+10

260

In [38]:
cube_file_1

'G250-260-12CO_giantcube_fullres.fits'

In [59]:
def extract_exceptions(cube_file, output_dir):
    # Open the spectral cube FITS file
    with fits.open(cube_file,ignore_missing_simple=True) as hdul:
        cube_data = hdul[0].data
        header = hdul[0].header
        wcs = WCS(header)

    # Create SQL engine and read coordinates -- crates data frame with all clumps within the cube
	# This is set for the 1 degree cubes. The range will need to be changed if bigger
    #df = pd.read_sql_query(f'SELECT gal_long, gal_lat, higal_name FROM higal_mopra_co WHERE gal_long>{int(cube_file[1:4])} and gal_long<{int(cube_file[1:4])+10}', con=mydb)
    df = pd.read_sql_query(f'SELECT gal_long, gal_lat, higal_name FROM higal_mopra_co WHERE higal_name="HIGALBM259.9911-0.2148"', con=mydb)

    for index, row in df.iterrows():
        # Convert galactic coordinates to pixel coordinates
        sky_coord = SkyCoord(l=row['gal_long']*u.deg, b=row['gal_lat']*u.deg, frame='galactic')
        x, y = sky_coord.to_pixel(wcs, origin=0)

        # Define the 3x3 pixel area around the source
        x_min_spec, x_max_spec = int(x) - 1, int(x) + 1
        y_min_spec, y_max_spec = int(y) - 1, int(y) + 1

        # Ensure we don't go out of bounds for the spectra
        x_min_spec = max(x_min_spec, 0)
        y_min_spec = max(y_min_spec, 0)
        x_max_spec = min(x_max_spec, cube_data.shape[2] - 1)
        #y_max_spec = min(y_max_spec, cube_data.shape[1] - 1)

        # Extract the smaller cube for the spectra
        subcube_spec = cube_data[:, y_min_spec:y_max_spec+1, x_min_spec:x_max_spec+1]

        # Compute the spectra (average over the 3x3 area)
        spectra = np.mean(subcube_spec, axis=(1, 2))

        # Create a new FITS file for the spectra
        hdu_spec = fits.PrimaryHDU(spectra, header)
        output_path_spec = f"{output_dir}/12CO_{row['higal_name']}_spec.fits"
        hdu_spec.writeto(output_path_spec, overwrite=True)
        #print(f"Saved spectra to {output_path_spec}")


In [60]:
extract_exceptions(cube_file_1,'/Users/kyriakos.trakakis/Desktop/Temp/Exceptions')

/var/folders/my/spqrhq7n74bcr2n7xfhc9hhr0000gp/T/ipykernel_87379/794008913.py:11: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  df = pd.read_sql_query(f'SELECT gal_long, gal_lat, higal_name FROM higal_mopra_co WHERE higal_name="HIGALBM259.9911-0.2148"', con=mydb)


In [67]:
with fits.open(cube_file_1,ignore_missing_simple=True) as hdul:
    cube_data = hdul[0].data
    header = hdul[0].header
    wcs = WCS(header)

    # Create SQL engine and read coordinates -- crates data frame with all clumps within the cube
	# This is set for the 1 degree cubes. The range will need to be changed if bigger ---------------v-----------------------------------------------------v
    #df = pd.read_sql_query(f'SELECT gal_long, gal_lat, higal_name FROM higal_mopra_co WHERE gal_long>{int(cube_file[1:4])} and gal_long<{int(cube_file[1:4])+10}', con=mydb)
    df = pd.read_sql_query(f'SELECT gal_long, gal_lat, higal_name FROM higal_mopra_co WHERE higal_name="HIGALBM259.9911-0.2148"', con=mydb)

    for index, row in df.iterrows():
        # Convert galactic coordinates to pixel coordinates
        sky_coord = SkyCoord(l=row['gal_long']*u.deg, b=row['gal_lat']*u.deg, frame='galactic')
        x, y = sky_coord.to_pixel(wcs, origin=0)

/var/folders/my/spqrhq7n74bcr2n7xfhc9hhr0000gp/T/ipykernel_87379/3014372355.py:9: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  df = pd.read_sql_query(f'SELECT gal_long, gal_lat, higal_name FROM higal_mopra_co WHERE higal_name="HIGALBM259.9911-0.2148"', con=mydb)


In [80]:
x_min_spec=int(x)-1
x_min_spec = max(x_min_spec, 0)
x_min_spec

6

In [81]:
x_max_spec=int(x)+1
x_max_spec = min(x_max_spec, cube_data.shape[2] - 1)
x_max_spec

8

In [82]:
cube_data.shape[2]

1213

In [86]:
cube_data.shape

(11139, 263, 1213)